In [1]:
# script to filter empirical mpra data for figure 1 malinois correlation scatterplots
# raw mpra data initially filtered for best library, and empirical emvars were called
# R script 'MPRA_Data_Filtering.R' handles that filtering

# this notebook further filters that data to drop:
# high standard error positions, ref | alt SE > 1
# variants whose ref/alt designation changes between hg19 and hg38
# variants whose sequences do not match between hg19 and hg38
# variants that change chromosome between hg19 and hg38
# variants that fail liftover between hg19 and hg38
# variants that are shared between ukbb/bbj (traits) and gtex are averaged to give single measure
# all variants are snps

In [12]:
# import packages
import pandas as pd
from tqdm import tqdm

In [ ]:
# define function for filtering on l2fc_SE
def l2fc_se_filter (df, filter_cutoff):
    # filter df for l2fc_se < 1 in both ref and alt
    filter_df = df[(df['A_log2FC_SE'] < filter_cutoff) &
                   (df['B_log2FC_SE'] < filter_cutoff)]
    return filter_df

In [9]:
# define function to add  malinois cell types - ie K562, HepG2, and SK-N-SH
def malinois_any_emvar (df):
    # subset df for only K562, HepG2, and SK-N-SH
    mal_cells = df[(df['cell_type'] == 'K562') | 
                   (df['cell_type'] == 'SKNSH') | 
                   (df['cell_type'] == 'HEPG2')].drop_duplicates(subset='A_log2FC')
    # make list of emVar IDs
    mal_emvar_ids = [j for i,j in zip(mal_cells['emVar'],
                                      mal_cells['ID']) if i == True]
    # make a dictionary of ID : emVar True pairs
    mal_emvar_dict = dict(zip(mal_emvar_ids,
                              [True for i in range(len(mal_emvar_ids))]))
    # add malinois emvar column to df
    df['malinois_emVar'] = [mal_emvar_dict.get(i) if i in mal_emvar_dict.keys() else False for i in tqdm(df['ID'])]
    return df

In [6]:
# open output of R script
# GTEx
gtex_mpra = pd.read_csv('/Users/buttsj/Dropbox (JAX)/Variant_Effects/ukbb_gtex_mpra/final_datasets/gtex_mpra_paired_filtered_emvar.txt',
                        sep = '\t',
                       low_memory=False)
# Traits
traits_mpra = pd.read_csv('/Users/buttsj/Dropbox (JAX)/Variant_Effects/ukbb_gtex_mpra/final_datasets/traits_mpra_paired_filtered_emvar.txt',
                          sep = '\t',
                         low_memory=False)

In [7]:
# filter gtex on l2fc_se
gtex_mpra = l2fc_se_filter(gtex_mpra, 
                           1)
# filter traits on l2fc_se
traits_mpra = l2fc_se_filter(traits_mpra, 
                             1)

In [13]:
# add column for calling emVars in only Malinois Cell Types
# gtex
gtex_mpra = malinois_any_emvar(gtex_mpra)
# add 'hg38_id' column to gtex for matching to predictions
gtex_mpra['hg38_id'] = [(':').join([i.split('_')[0], 
                                    i.split('_')[1],
                                    i.split('_')[2], 
                                    i.split('_')[3]]) for i in gtex_mpra['variant_hg38']]
# traits
traits_mpra = malinois_any_emvar(traits_mpra)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1108490/1108490 [00:00<00:00, 6720773.94it/s]


In [29]:
# traits
# Add hg38 Positions to MPRA Data
# 110 IDs failed liftover - in 'traits_hg19_unmapped'
# open lifted IDs
hg38_ids_from_liftover = pd.read_csv('/Users/buttsj/Dropbox (JAX)/Variant_Effects/ukbb_gtex_mpra/ukbb_liftover/traits_bed_hg38.bed',
                                    sep = '\t',
                                    header=None)
# filter full traits data for only those IDs that lifted successfully
traits_mpra = traits_mpra[traits_mpra['variant'].isin(hg38_ids_from_liftover[3])]
# make list of hg38 IDs
hg38_id = [(':').join([i, str(j), k.split(':')[-2], k.split(':')[-1]]) for i,j,k in zip(hg38_ids_from_liftover[0],
                                                                                        hg38_ids_from_liftover[1],
                                                                                        hg38_ids_from_liftover[3])]
# make a dictionary of hg19 ID : hg38 ID pairs
hg19_hg38_id_dict = dict(zip(hg38_ids_from_liftover[3],
                            hg38_id))
# match hg19 and hg38 ids to add column to DF
traits_mpra['hg38_id'] = [hg19_hg38_id_dict.get(i) for i in traits_mpra['variant']]

In [24]:
### Final Filtering ###
# Open GTEx Filtering Data
# Open hg19-hg38 Ref | Alt Flips, defined in hg19_hg38_mismatch_summary.py
gtex_hg19_hg38_ref_alt_flips = pd.read_csv('/Users/buttsj/Dropbox (JAX)/Variant_Effects/ukbb_gtex_mpra/ukbb_gtex_mpra_chrom_vcfs/gtex/hg19_hg38_ref_alt_flip_ids.txt',
                                           sep = '\t')['hg19 id'].tolist()
# Open hg19-hg38 Chromosome Moves, defined in blacklist_outline.py
gtex_chrom_chrom_move_ids = pd.read_csv('/Users/buttsj/Dropbox (JAX)/Variant_Effects/ukbb_gtex_mpra/empirical_ukbb_gtex/chrom_move_ids.txt',
                                        sep = '\t')['ID'].tolist()
# open liftover fails
gtex_hg19_liftover_fails = pd.read_csv('/Users/buttsj/Downloads/gtex_hg19_liftover_fails.txt',
                                       sep =  '\t',
                                       header=None)[0].tolist()

In [25]:
# Open Traits Filtering Data
# check chromosome concordance between hg19 and hg38 coordinates in Traits Data
traits_mismatch_chroms = list(pd.Series([k for i,j,k in zip(traits_mpra['hg38_id'],
                                                            traits_mpra['variant'],
                                                            traits_mpra['ID']) if i.split(':')[0] != j.split(':')[0]]).unique())
# open liftover fails
traits_liftover_fails = pd.read_csv('/Users/buttsj/Dropbox (JAX)/Variant_Effects/ukbb_gtex_mpra/ukbb_liftover/hg38_traits_for_seq_check__mismatches.txt',
                                    sep = '\t',
                                    header=None)[0].tolist()

In [26]:
# open 1kg hg38 IDs
oneKG_IDs = pd.read_csv('/Users/buttsj/Dropbox (JAX)/Variant_Effects/ukbb_gtex_mpra/ukbb_liftover/1kg_hg38_ids.txt',
                        sep = '\t')
# make a dictionary of onKG_IDs
oneKG_dict = dict(zip(oneKG_IDs['1kg_hg38_vcf_id'],
                      oneKG_IDs['1kg_hg38_vcf_id']))
matched_ids = []
missed_ids = []
for i in tqdm(traits_mpra['hg38_id'].unique()):
    if i in oneKG_dict.keys():
        matched_ids.append(oneKG_dict.get(i))
    else:
        missed_ids.append(i)
# ~6K have no match in 1KG hg38 IDs
# save missed IDs to disk as text file for checking
oneKG_missed_df = pd.DataFrame({'ID' : missed_ids})
# combine blacklist into one complete list for filtering MPRA data
blacklist_all = gtex_hg19_hg38_ref_alt_flips + gtex_chrom_chrom_move_ids  + gtex_hg19_liftover_fails + traits_mismatch_chroms + traits_liftover_fails + missed_ids
blacklist_unique = list(pd.Series(blacklist_all).unique())
# Total of 33759 blacklisted IDs

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126044/126044 [00:00<00:00, 1114200.26it/s]


In [31]:
# Total of 33759 blacklisted IDs
# filter gtex and traits for blacklist IDs
# gtex
gtex_blacklist_filter = gtex_mpra[~gtex_mpra['ID'].isin(blacklist_unique)]
# traits
traits_blacklist_filter = traits_mpra[~traits_mpra['ID'].isin(blacklist_unique)]

In [32]:
# Get IDs that are shared between gtex and traits
shared_ids = []
for i in tqdm(gtex_blacklist_filter['ID'].unique()):
    if i in traits_blacklist_filter['ID'].tolist():
        shared_ids.append(i)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 181607/181607 [33:48<00:00, 89.52it/s]


In [34]:
# Combine Datasets into Single DataFrame
# get shared column names
shared_columns = [i for i in gtex_blacklist_filter.keys() if i in traits_blacklist_filter.keys()]
# filter for only shared columns for concatenating DF
# gtex
gtex_blacklist_filter = gtex_blacklist_filter.filter(shared_columns)
# traits
traits_blacklist_filter = traits_blacklist_filter.filter(shared_columns)
# concatenate DFs into single
all_blacklist_filter_mpra = pd.concat([gtex_blacklist_filter, traits_blacklist_filter])

In [40]:
# Open Average reduced shared IDs
avg_shared = pd.read_csv('/Users/buttsj/Dropbox (JAX)/Variant_Effects/ukbb_gtex_mpra/final_datasets/tmp/average_reduced_shared_mpra_data.txt',
                            sep = '\t')
# Drop shared IDs from full dataset
no_shared_ids = all_blacklist_filter_mpra[~all_blacklist_filter_mpra['ID'].isin(shared_ids)]
# concatenate avg reduced and dataset without shared IDs
final_mpra_df = pd.concat([avg_shared, no_shared_ids])
# save blacklist filtered, shared ID reduced DF to file for all correlations, etc.
final_mpra_df.to_csv('/Users/buttsj/Dropbox (JAX)/Variant_Effects/ukbb_gtex_mpra/final_datasets/all.gtex.traits.mpra.blacklist.filtered.v1.txt',
                     sep = '\t',
                     index=False)

/var/folders/58/97_f_jfx1yg65t5hxh4pl0pwn14x9s/T/ipykernel_2985/1011778812.py:2: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  avg_shared = pd.read_csv('/Users/buttsj/Dropbox (JAX)/Variant_Effects/ukbb_gtex_mpra/final_datasets/tmp/average_reduced_shared_mpra_data.txt',
